# 第 18 课｜温度、Top-k、Top-p 与生成评测

**固定同一个微型 GPT，改变“怎样选下一个 token”，观察生成分布、文本和评测指标。**

上一课实现了模型、训练和基础生成。这一课不再添加网络层，而是把模型输出后的选择规则做清楚。

| 材料与环境 | 约定 |
|---|---|
| 交付 | 本 Notebook + `lesson_18_decoding_and_generation_evaluation.py`，互不依赖 |
| 运行 | 沿用 PyTorch；CPU 足够，不下载权重或语料 |
| 有第 17 课检查点 | 设置 `CHECKPOINT` 后直接加载，跳过训练 |
| 没有检查点 | 内置同结构模型及模板数据，先训练一次，再固定模型 |
| 实验范围 | 小模型模板续写，不是通用中文、事实性、推理或编程基准 |

**解码改变被抽取的输出，不会在本课中更新模型参数。多样性、格式正确性、事实正确性也不是同一个指标。**

## 学习顺序与验收

先用四个候选的手算分布学习温度与过滤，再连接真实模型，最后阅读完整评测结果。
本课应能独立完成：解释 `logits / temperature`；实现 Top-k 和 Top-p；说明过滤顺序；按同一提示与多个 seed 比较输出；辨别多样性和任务成功率。

代码分为**新知识**和**前课组件复用**。后者提供独立运行所需的定义，可以先折叠阅读。
练习区有独立开关；运行全部单元成功，不代表已经亲手完成练习。参考答案集中放在后面。

## 0. 环境与约定

继续使用能运行第 17 课的 Python/PyTorch 环境；没有新增必装包，也不需要 Transformers。
尚未安装 PyTorch 时查阅 [官方安装入口](https://pytorch.org/get-started/locally/)，不要为本课覆盖已经可用的 CUDA 环境。

本课用 `tau` 或 `temperature` 表示温度，避免与序列长度 `T` 混淆。
所有随机数实验只保证当前软件与设备环境下的可复现性；不同版本或 CPU/CUDA 的输出不保证逐位一致。[4]

In [1]:
from __future__ import annotations

import argparse
from dataclasses import asdict, dataclass
from functools import partial
import hashlib
import json
import math
from pathlib import Path
import random
import tempfile

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset



SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNK_ID, BOS_ID, EOS_ID, PAD_ID = range(4)
IGNORE_INDEX = -100

def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")


def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)


def set_seed(seed: int) -> None:
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class GPTConfig:
    vocab_size: int
    max_len: int = 64
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    ff_dim: int = 256
    dropout: float = 0.1

    def __post_init__(self) -> None:
        for name in ("vocab_size", "max_len", "d_model", "n_heads", "n_layers", "ff_dim"):
            positive_int(getattr(self, name), name)
        if self.vocab_size <= len(SPECIAL_TOKENS):
            raise ValueError("词表除了 4 个特殊 token，还必须有普通字符")
        if self.d_model % self.n_heads:
            raise ValueError("d_model 必须能被 n_heads 整除")
        if (isinstance(self.dropout, bool) or not isinstance(self.dropout, (int, float))
                or not math.isfinite(self.dropout) or not 0 <= self.dropout < 1):
            raise ValueError("dropout 必须在 [0,1) 内")

In [2]:
torch.set_num_threads(1)
DEVICE = choose_device("cpu")  # 已有 CUDA 环境可改为 "auto"
print("PyTorch：", torch.__version__)
print("设备：", DEVICE)
print("CUDA 可用：", torch.cuda.is_available())
print("当前工作目录：", Path.cwd())

PyTorch： 2.10.0+cpu
设备： cpu
CUDA 可用： False
当前工作目录： /mnt/data/_lesson18_build/notebook_run


## 1. 模型分布与选择策略是两层东西

固定前缀 $x_{<t}$，模型给出 logits $z=f_\theta(x_{<t})$。原始模型分布是 $p_\theta=\operatorname{softmax}(z)$。
解码策略根据它决定下一 token；改变策略，不等于更新 $\theta$。[1]

**贪心**每步取最大分数；**采样**按概率随机抽取。[1]
贪心只是局部选择，不保证整段序列概率最高：若第一步 A/B 概率为 0.6/0.4，之后各自最优延续概率为 0.51/0.99，
贪心路径概率为 $0.6\times0.51=0.306$，而另一条可达 $0.4\times0.99=0.396$。
这也不意味着最高概率文本就一定事实正确。

下面先用四个候选 A/B/C/D，不运行神经网络。

In [3]:
@dataclass(frozen=True)
class DecodeConfig:
    mode: str = "sample"
    temperature: float = 1.0
    top_k: int = 0                 # 0 表示不使用 Top-k
    top_p: float = 1.0             # 1 表示不使用 Top-p

    def __post_init__(self) -> None:
        if self.mode not in {"greedy", "sample"}:
            raise ValueError("mode 必须是 greedy 或 sample")
        if (isinstance(self.temperature, bool) or not isinstance(self.temperature, (int, float))
                or not math.isfinite(self.temperature) or self.temperature <= 0):
            raise ValueError("temperature 必须是有限正数；贪心使用 mode='greedy'，不要除以零")
        if type(self.top_k) is not int or self.top_k < 0:
            raise ValueError("top_k 必须是非负整数；0 表示禁用")
        if (isinstance(self.top_p, bool) or not isinstance(self.top_p, (int, float))
                or not math.isfinite(self.top_p) or not 0 < self.top_p <= 1):
            raise ValueError("top_p 必须在 (0,1] 内；1 表示禁用")
        if self.mode == "greedy" and (self.temperature != 1.0 or self.top_k != 0 or self.top_p != 1.0):
            raise ValueError("本课贪心模式不接收采样调节参数，请保持 temperature=1、top_k=0、top_p=1")


def validate_logits(logits: torch.Tensor) -> torch.Tensor:
    """允许 -inf 表示被屏蔽；拒绝 NaN、+inf、全屏蔽行。返回副本，不修改调用者。"""
    if (not isinstance(logits, torch.Tensor) or not logits.is_floating_point()
            or logits.ndim not in (1, 2) or min(logits.shape) <= 0):
        raise ValueError("logits 必须是非空浮点 [V] 或 [B,V] 张量")
    if bool(torch.isnan(logits).any()) or bool(torch.isposinf(logits).any()):
        raise ValueError("logits 不允许包含 NaN 或正无穷")
    if not bool(torch.isfinite(logits).any(dim=-1).all()):
        raise ValueError("每行至少需要一个有限候选，不能把整行屏蔽")
    return logits.to(dtype=torch.float64).clone()


def ban_tokens(logits: torch.Tensor, forbidden_ids: tuple[int, ...] = ()) -> torch.Tensor:
    z = validate_logits(logits)
    if not isinstance(forbidden_ids, (list, tuple)):
        raise ValueError("forbidden_ids 必须是编号列表或元组")
    if any(type(i) is not int or not 0 <= i < z.shape[-1] for i in forbidden_ids):
        raise ValueError("禁止编号必须是词表范围内的整数")
    if forbidden_ids:
        z[..., list(forbidden_ids)] = float("-inf")
    return validate_logits(z)

In [4]:
toy_tokens = ["A", "B", "C", "D"]
base_probs = torch.tensor([0.5, 0.3, 0.15, 0.05], dtype=torch.float64)
toy_logits = base_probs.log()  # softmax(log p) 恢复归一化后的 p
print("原始概率：", base_probs.tolist())
print("贪心选择：", toy_tokens[toy_logits.argmax().item()])
print("贪心路径与另一条路径：", 0.6 * 0.51, 0.4 * 0.99)

原始概率： [0.5, 0.3, 0.15, 0.05]
贪心选择： A
贪心路径与另一条路径： 0.306 0.396


## 2. 温度：重新分配概率，不重新训练

$$p_i^{(\tau)}=\frac{\exp(z_i/\tau)}{\sum_j\exp(z_j/\tau)},\qquad \tau>0$$

$\tau<1$ 时概率更集中；$\tau>1$ 时更平坦；正温度不改变 logits 的大小顺序。[2]
在同一有限候选集合、同一固定前缀下，可以用熵 $H(p)=-\sum_i p_i\log p_i$ 检查分散程度。
**沿不同生成轨迹的平均熵、不同文本比例，不保证随温度单调变化**：前一步抽取不同，下一步的前缀和 logits 就变了。

数值实现先减最大值，再除温度。减去同一个常数不改变 softmax。
温度为零不能直接代入公式；本课用显式 `mode="greedy"` 表示贪心。

In [5]:
def temperature_scale(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    DecodeConfig(temperature=temperature)
    z = validate_logits(logits)
    # 减最大值不改变 softmax，先减再除可避免大的正 logits 除小温度溢出。
    return (z - z.max(dim=-1, keepdim=True).values) / temperature


def distribution_entropy(probabilities: torch.Tensor) -> float:
    """一维分布熵，单位 nat；只对正概率取 log，避免 0*log(0)。"""
    if (probabilities.ndim != 1 or not bool(torch.isfinite(probabilities).all())
            or bool((probabilities < 0).any())):
        raise ValueError("概率需要是一维非负有限向量")
    torch.testing.assert_close(probabilities.sum(), probabilities.new_tensor(1.), rtol=1e-6, atol=1e-8)
    positive = probabilities[probabilities > 0]
    return float(-(positive * positive.log()).sum().item())

In [6]:
for tau in (0.5, 1.0, 2.0):
    p = torch.softmax(temperature_scale(toy_logits, tau), dim=-1)
    print(f"tau={tau}: 概率={[round(v, 4) for v in p.tolist()]}，熵={distribution_entropy(p):.4f}")
# 先预测：三个温度下的最大概率 token 会不会变？

tau=0.5: 概率=[0.6849, 0.2466, 0.0616, 0.0068]，熵=0.8103
tau=1.0: 概率=[0.5, 0.3, 0.15, 0.05]，熵=1.1421
tau=2.0: 概率=[0.379, 0.2936, 0.2076, 0.1198]，熵=1.3082


## 3. Top-k：按排名保留候选，再重新归一化

取概率最大的 k 个候选，其他分数设为负无穷，再做 softmax。[2]
这里 k=2 时保留 A/B，新的概率不是 [0.5,0.3,0,0]，而是 [0.625,0.375,0,0]。

本课约定 `k=0` 禁用；`k>=词表大小` 保留全部；已有屏蔽候选不会复活。
为方便测试，使用稳定排序精确保留 k 个可选索引，分数并列时较小编号优先。
**这是本课的 tie-breaking 约定，不能假设所有库都完全一样。**有些阈值式实现会保留所有并列候选。

In [7]:
def top_k_filter(logits: torch.Tensor, k: int = 0) -> torch.Tensor:
    DecodeConfig(top_k=k)
    z = validate_logits(logits)
    if k == 0 or k >= z.shape[-1]:
        return z
    # 精确选择 k 个索引；稳定排序规定并列时较小 token ID 优先。
    indices = torch.argsort(z, dim=-1, descending=True, stable=True)[..., :k]
    filtered = torch.full_like(z, float("-inf"))
    filtered.scatter_(-1, indices, z.gather(-1, indices))
    return filtered

In [8]:
filtered_k = top_k_filter(toy_logits, 2)
print("过滤后的分数：", filtered_k.tolist())
print("重归一化概率：", torch.softmax(filtered_k, -1).tolist())
print("k=1：", torch.softmax(top_k_filter(toy_logits, 1), -1).tolist())
print("并列时保留：", torch.isfinite(top_k_filter(torch.zeros(4), 2)).tolist())

过滤后的分数： [-0.6931471805599453, -1.2039728043259361, -inf, -inf]
重归一化概率： [0.625, 0.3749999999999999, 0.0, 0.0]
k=1： [1.0, 0.0, 0.0, 0.0]
并列时保留： [True, True, False, False]


## 4. Top-p / Nucleus Sampling：累计概率决定集合大小

Top-p 将候选按概率从高到低排列，保留累计概率**首次达到或超过 p**的最小前缀，再归一化。[2][3]
它不是“保留 p 比例的 token”，也不是“保留单个概率超过 p 的 token”。

以原始分布和 p=0.7 为例：

| 候选 | 概率 | 加入后的累计概率 | 保留？ |
|---|---:|---:|---|
| A | 0.50 | 0.50 | 是 |
| B | 0.30 | 0.80 | 是，首次跨过阈值 |
| C | 0.15 | 0.95 | 否 |
| D | 0.05 | 1.00 | 否 |

**跨过阈值的 B 必须保留。**直接写 `累计概率 <= p` 会错误丢掉它。
一种容易解释的写法是：如果“加入当前候选之前”的累计概率仍小于 p，就保留当前候选。
本课对恰好达到 p 的情况不再多留一个；至少保留一个候选，p=1 显式禁用。边界受浮点表示影响，不应靠肉眼估计。

In [9]:
def top_p_filter(logits: torch.Tensor, p: float = 1.0) -> torch.Tensor:
    DecodeConfig(top_p=p)
    z = validate_logits(logits)
    if p == 1.0:
        return z
    sorted_logits, sorted_indices = torch.sort(z, dim=-1, descending=True, stable=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative = sorted_probs.cumsum(dim=-1)
    # 候选自身之前的累计概率 < p，就必须保留它。
    # 不能使用 cumulative <= p：那样会丢掉首次跨过阈值的那个候选。
    previous_mass = torch.cat([torch.zeros_like(cumulative[..., :1]), cumulative[..., :-1]], dim=-1)
    keep = (previous_mass < p) & torch.isfinite(sorted_logits)
    keep[..., 0] = True
    sorted_filtered = sorted_logits.masked_fill(~keep, float("-inf"))
    result = torch.full_like(z, float("-inf"))
    result.scatter_(-1, sorted_indices, sorted_filtered)
    return result

In [10]:
for threshold in (0.2, 0.7, 0.9, 1.0):
    kept = top_p_filter(toy_logits, threshold)
    print("p=", threshold,
          "保留=", [t for t, flag in zip(toy_tokens, torch.isfinite(kept).tolist()) if flag],
          "概率=", [round(v, 4) for v in torch.softmax(kept, -1).tolist()])
print("恰好到达0.5：", torch.softmax(top_p_filter(torch.zeros(2), 0.5), -1).tolist())

p= 0.2 保留= ['A'] 概率= [1.0, 0.0, 0.0, 0.0]
p= 0.7 保留= ['A', 'B'] 概率= [0.625, 0.375, 0.0, 0.0]
p= 0.9 保留= ['A', 'B', 'C'] 概率= [0.5263, 0.3158, 0.1579, 0.0]
p= 1.0 保留= ['A', 'B', 'C', 'D'] 概率= [0.5, 0.3, 0.15, 0.05]
恰好到达0.5： [1.0, 0.0]


## 5. 组合顺序会改变结果

本课把顺序写死并记录在脚本说明中：

**禁止 BOS/PAD → 温度 → Top-k → Top-p → softmax → 抽样。**

Top-p 使用 Top-k 之后重新归一化的概率。
以 [0.5,0.3,0.15,0.05]、k=2、p=0.6 为例：先 k 后 p 时第一项变成 0.625，所以只保留 A；
若先 p 后 k，原始第一项 0.5 尚未达到 0.6，会保留 A/B。两者不同。

先学习每个单独参数，最后才研究组合。不要把不同框架或 API 的默认值、过滤顺序视为同一约定。
过滤函数支持 [V]、[B,V]；小词表使用 float64 便于核对边界，这不是面向大型模型的吞吐优化。

In [11]:
def sampling_probs(logits: torch.Tensor, *, temperature: float = 1.0, top_k: int = 0,
                   top_p: float = 1.0, forbidden_ids: tuple[int, ...] = ()) -> torch.Tensor:
    """默认顺序是本课的明确约定；Top-p 使用 Top-k 后重新归一化的分布。"""
    z = ban_tokens(logits, forbidden_ids)
    z = temperature_scale(z, temperature)
    z = top_k_filter(z, top_k)
    z = top_p_filter(z, top_p)
    probabilities = torch.softmax(z, dim=-1)
    if not bool(torch.isfinite(probabilities).all()):
        raise ValueError("产生了非有限概率，请检查输入数值")
    return probabilities

In [12]:
print("先k后p：", sampling_probs(toy_logits, top_k=2, top_p=.6).tolist())
reverse = top_k_filter(top_p_filter(toy_logits, .6), 2)
print("先p后k：", torch.softmax(reverse, -1).tolist())
for tau in (.4, 1., 2.):
    p = sampling_probs(toy_logits, temperature=tau, top_p=.7)
    print("tau=", tau, "候选数=", int((p>0).sum()), "概率=", p.tolist())
print("原始logits未修改：", torch.equal(toy_logits, base_probs.log()))

先k后p： [1.0, 0.0, 0.0, 0.0]
先p后k： [0.625, 0.3749999999999999, 0.0, 0.0]
tau= 0.4 候选数= 1 概率= [1.0, 0.0, 0.0, 0.0]
tau= 1.0 候选数= 2 概率= [0.625, 0.3749999999999999, 0.0, 0.0]
tau= 2.0 候选数= 3 概率= [0.4306040222561933, 0.33354444140163286, 0.2358515363421737, 0.0]
原始logits未修改： True


## 6. 真正抽样：局部随机源，而不是反复改全局种子

`torch.multinomial` 根据非负有限权重抽样；每行权重总和必须大于零。[5]
单步生成只抽一个；对同一分布重复实验时可用 `replacement=True`。

`torch.Generator(...).manual_seed(seed)` 给一次生成独立的随机源。同环境、同模型、同策略、同提示和同 seed 应当复现；
不同 seed 可能仍抽到相同输出，相同 seed 用于不同分布也不保证选中相同 token。跨设备/版本不承诺逐位一致。[4]

贪心模式不需要随机抽样。本课拒绝给贪心模式传非默认采样参数，避免“调了但没生效”的误解。

In [13]:
def choose_next(logits: torch.Tensor, config: DecodeConfig,
                generator: torch.Generator | None = None,
                forbidden_ids: tuple[int, ...] = (BOS_ID, PAD_ID)) -> tuple[int, dict]:
    """单个位置 [V]；返回 token ID 与用于观察的分布信息。"""
    if not isinstance(config, DecodeConfig):
        raise ValueError("config 必须是 DecodeConfig")
    z = ban_tokens(logits, forbidden_ids)
    if z.ndim != 1:
        raise ValueError("choose_next 一次只接收一个位置的 [V] logits")
    if config.mode == "greedy":
        next_id = int(z.argmax().item())
        probabilities = torch.zeros_like(z)
        probabilities[next_id] = 1.0
    else:
        probabilities = sampling_probs(z, temperature=config.temperature,
                                       top_k=config.top_k, top_p=config.top_p)
        next_id = int(torch.multinomial(probabilities, 1, generator=generator).item())
    info = {"support_size": int((probabilities > 0).sum().item()),
            "entropy_nats": distribution_entropy(probabilities),
            "chosen_probability": float(probabilities[next_id].item())}
    return next_id, info

In [14]:
g1 = torch.Generator().manual_seed(18)
g2 = torch.Generator().manual_seed(18)
draw1 = torch.multinomial(base_probs, 12, replacement=True, generator=g1)
draw2 = torch.multinomial(base_probs, 12, replacement=True, generator=g2)
print("抽样序列：", [toy_tokens[i] for i in draw1.tolist()])
print("同seed是否一致：", torch.equal(draw1, draw2))
# 只有4个普通候选的实验，不能把第17课特殊token编号套到它上面。
print("贪心：", choose_next(toy_logits, DecodeConfig(mode="greedy"), forbidden_ids=()))

抽样序列： ['D', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'D', 'D', 'A']
同seed是否一致： True
贪心： (0, {'support_size': 1, 'entropy_nats': -0.0, 'chosen_probability': 1.0})


## 7. 复用第 17 课的模型、数据与检查点

下面几格只是为了让讲义独立运行。网络结构、参数名称、默认训练配方与第 17 课一致：
两层 Pre-LN、D=64、4 个头、上下文 64、独立输出头。你可以先折叠这些实现，重点看后面的生成流程。

训练数据在内存生成，不会创建或修改前课的 `corpus.txt`。
自定义语料训练出的第17课模型也可载入，但需要传词表能表示的提示；自定义提示不计算内置模板成功率。

In [15]:
def demo_lines() -> list[str]:
    """96 条原创模板句；目的只是验证训练流程，不是中文能力基准。"""
    subjects = ["我", "你", "小明", "小红", "同学", "老师", "我们", "他们"]
    times = ["今天", "明天"]
    activities = ["学习数学", "编写程序", "阅读故事", "整理笔记", "检查数据", "训练模型"]
    return [s + t + "正在" + a + "。" for s in subjects for t in times for a in activities]


def clean_texts(lines: list[str]) -> list[str]:
    if not isinstance(lines, list) or any(not isinstance(t, str) for t in lines):
        raise ValueError("语料必须是字符串列表")
    texts, seen = [], set()
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts


def split_texts(texts: list[str], seed: int = 17) -> tuple[list[str], list[str]]:
    if (not isinstance(texts, list) or len(texts) < 5
            or any(not isinstance(t, str) or not t for t in texts)):
        raise ValueError("清理后至少需要 5 条不同的非空文本")
    if len(set(texts)) != len(texts):
        raise ValueError("请先按完整文本去重，再切分")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, len(texts) // 5)
    return shuffled[n_val:], shuffled[:n_val]


def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    if not train_texts or any(not isinstance(t, str) for t in train_texts):
        raise ValueError("训练文本必须是非空字符串列表")
    # 排序固定编号顺序；只能使用训练侧字符。
    chars = sorted(set("".join(train_texts)))
    id_to_token = SPECIAL_TOKENS + chars
    return {token: i for i, token in enumerate(id_to_token)}, id_to_token


def validate_vocab(id_to_token: list[str]) -> None:
    if (not isinstance(id_to_token, list) or len(id_to_token) <= 4
            or id_to_token[:4] != SPECIAL_TOKENS
            or any(not isinstance(t, str) for t in id_to_token)
            or any(len(t) != 1 for t in id_to_token[4:])
            or len(set(id_to_token)) != len(id_to_token)):
        raise ValueError("词表必须按固定顺序包含四个特殊 token，之后是互不重复的单个字符")


def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, UNK_ID) for char in text]

In [16]:
class CharDataset(Dataset):
    def __init__(self, texts: list[str], token_to_id: dict[str, int], max_len: int) -> None:
        positive_int(max_len, "max_len")
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空文本列表")
        self.sequences = []
        for index, text in enumerate(texts):
            ids = [BOS_ID] + encode(text, token_to_id) + [EOS_ID]
            # 输入和目标比完整序列各少一位；不允许静默截断。
            if len(ids) - 1 > max_len:
                raise ValueError(f"第 {index + 1} 条文本需 {len(ids)-1} 个输入位置，超过 max_len={max_len}；请手动分段或调大上下文")
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        ids = self.sequences[index]
        return ids[:-1], ids[1:]


def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys = [], []
    for x, y in samples:
        if (x.ndim != 1 or x.numel() == 0 or x.shape != y.shape
                or x.dtype != torch.long or y.dtype != torch.long):
            raise ValueError("每条样本的输入、标签必须是等长非空 long 向量")
        xs.append(x)
        ys.append(y)
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=PAD_ID),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
    }


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = 17) -> DataLoader:
    positive_int(batch_size, "batch_size")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(seed))

In [17]:
def attention_allowed(key_valid: torch.Tensor) -> torch.Tensor:
    """[B,T] -> [B,1,T,T]；True=允许读取。只接受非空、右侧补齐。"""
    if (not isinstance(key_valid, torch.Tensor) or key_valid.dtype != torch.bool
            or key_valid.ndim != 2 or min(key_valid.shape) <= 0):
        raise ValueError("key_valid 必须是非空 [B,T] 布尔张量")
    if not bool(key_valid[:, 0].all()):
        raise ValueError("首位置必须有效；不接受全 PAD 或左侧 padding")
    if bool((~key_valid[:, :-1] & key_valid[:, 1:]).any()):
        raise ValueError("只接受连续有效前缀加右 padding，不接受内部空洞")
    t = key_valid.shape[1]
    causal = torch.ones(t, t, dtype=torch.bool, device=key_valid.device).tril()
    # 不屏蔽整行 query：补齐 query 可有数值，但其标签被忽略，避免 softmax 全 -inf。
    return causal[None, None, :, :] & key_valid[:, None, None, :]


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.n_heads = cfg.n_heads
        self.d_head = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attention_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        q = self.q_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        scores = scores.masked_fill(~allowed, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        heads = self.attention_dropout(weights) @ v
        merged = heads.transpose(1, 2).contiguous().reshape(b, t, d)
        return self.out_proj(merged)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(nn.Linear(cfg.d_model, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.d_model))
        self.residual_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        x = x + self.residual_dropout(self.attn(self.norm1(x), allowed))
        x = x + self.residual_dropout(self.ffn(self.norm2(x)))
        return x


class TinyGPT(nn.Module):
    """输入 [B,T]，输出 [B,T,V] 原始 logits。forward 不计算损失，不移位标签。"""
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(cfg.max_len, cfg.d_model)
        self.embedding_dropout = nn.Dropout(cfg.dropout)
        # 每次创建新的 Block；不要使用 [一个Block] * n_layers。
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        # 本课不共享输出头与输入 Embedding 的权重，方便辨认两个角色。
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        if (not isinstance(input_ids, torch.Tensor) or input_ids.dtype != torch.long
                or input_ids.ndim != 2 or min(input_ids.shape) <= 0):
            raise ValueError("input_ids 必须是非空 [B,T] torch.long 张量")
        if input_ids.device != self.token_embedding.weight.device:
            raise ValueError("输入与模型必须位于同一设备")
        if bool(((input_ids < 0) | (input_ids >= self.cfg.vocab_size)).any()):
            raise ValueError("输入包含词表范围之外的编号")
        t = input_ids.shape[1]
        if t > self.cfg.max_len:
            raise ValueError(f"输入长度 {t} 超过上下文 {self.cfg.max_len}；本课不自动截断")
        allowed = attention_allowed(input_ids != PAD_ID)
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        for block in self.blocks:
            x = block(x, allowed)
        return self.lm_head(self.final_norm(x))


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

In [18]:
def loss_parts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    if (logits.ndim != 3 or targets.shape != logits.shape[:2]
            or targets.dtype != torch.long or logits.device != targets.device):
        raise ValueError("需要同设备 logits=[B,T,V]、long targets=[B,T]")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits 包含非有限值")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("本批没有有效目标，不能计算平均损失")
    if bool(((targets[valid] < 0) | (targets[valid] >= logits.shape[-1])).any()):
        raise ValueError("有效目标编号越界")
    total = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                            ignore_index=IGNORE_INDEX, reduction="sum")
    return total, count


def finish_metrics(total_loss: float, total_correct: int, total_tokens: int) -> dict:
    if total_tokens <= 0:
        raise ValueError("没有可评估的有效 token")
    mean = total_loss / total_tokens
    if not math.isfinite(mean):
        raise ValueError("平均损失不是有限数")
    # JSON 不能保存 Inf；异常大损失时不报告 PPL 数值。
    ppl = math.exp(mean) if mean < 700 else None
    return {"loss": mean, "perplexity": ppl, "accuracy": total_correct / total_tokens,
            "tokens": total_tokens}


def evaluate(model: TinyGPT, loader: DataLoader, device: torch.device) -> dict:
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        model.eval()
        with torch.no_grad():
            for batch in loader:
                inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
                logits = model(inputs)
                loss_sum, count = loss_parts(logits, targets)
                valid = targets != IGNORE_INDEX
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    return finish_metrics(total_loss, total_correct, total_tokens)


def train_epoch(model: TinyGPT, loader: DataLoader, optimizer: torch.optim.Optimizer,
                device: torch.device, max_grad_norm: float = 1.0) -> dict:
    model.train()
    total_loss, total_correct, total_tokens, steps = 0.0, 0, 0, 0
    for batch in loader:
        inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss_sum, count = loss_parts(logits, targets)
        (loss_sum / count).backward()
        # 裁剪发生在 backward 之后、step 之前；不是直接截断参数。
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        valid = targets != IGNORE_INDEX
        total_loss += loss_sum.detach().item()
        total_correct += int(((logits.detach().argmax(-1) == targets) & valid).sum().item())
        total_tokens += count
        steps += 1
    result = finish_metrics(total_loss, total_correct, total_tokens)
    result["steps"] = steps
    return result  # 在线指标来自不断改变的参数；选模型用下面的固定参数评估。


def snapshot_state(model: nn.Module) -> dict[str, torch.Tensor]:
    # clone 防止“最佳参数”引用继续被后续训练修改。
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def save_checkpoint(path: Path, model: TinyGPT, id_to_token: list[str], best_epoch: int) -> None:
    validate_vocab(id_to_token)
    if len(id_to_token) != model.cfg.vocab_size:
        raise ValueError("模型与词表大小不一致")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({"format_version": 1, "config": asdict(model.cfg), "vocab": id_to_token,
                "model_state": snapshot_state(model), "best_epoch": best_epoch,
                "purpose": "inference_only_not_exact_training_resume"}, path)


def load_checkpoint(path: Path, device: torch.device) -> tuple[TinyGPT, list[str], dict]:
    # 只加载自己生成/信任的文件；weights_only 也不是任意文件安全性的保证。
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("format_version") != 1:
        raise ValueError("不支持的检查点格式")
    vocab = payload["vocab"]
    validate_vocab(vocab)
    cfg = GPTConfig(**payload["config"])
    if len(vocab) != cfg.vocab_size:
        raise ValueError("检查点词表与配置不一致")
    # 构造随机初始化模型不消耗调用者的 CPU 随机状态。
    with torch.random.fork_rng(devices=[]):
        model = TinyGPT(cfg)
    model.load_state_dict(payload["model_state"], strict=True)
    model.to(device).eval()
    return model, vocab, {"best_epoch": payload["best_epoch"], "purpose": payload["purpose"]}

## 8. 把采样器接回自回归生成

每一步只用 `model(input_ids)[0,-1,:]`，按照策略选编号，再追加到前缀。
本课禁止输出 BOS/PAD；EOS 可正常被选中并结束生成，UNK 如被选中会明确显示。

工程约定：
- `max_new_tokens` 只计算新增编号，包括 EOS；0 表示不调用模型。
- 提示字符必须在词表中；超长提示报错，不静默替换或截断。
- 最多输入 max_len 个 token，这时仍可预测下一个；若继续需要超长前向，停止并记录 `context_limit`。
- 保存并恢复各子模块模式，使用 `no_grad()`，不创建或清空已有参数梯度。

目前仍**没有 KV cache**，每一步重算整个前缀。本课不是推理速度测评。

In [19]:
def generate_text(model: TinyGPT, vocab: list[str], prompt: str, config: DecodeConfig,
                  *, max_new_tokens: int = 24, seed: int = 0) -> dict:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size:
        raise ValueError("词表与模型大小不一致")
    if not isinstance(prompt, str) or not isinstance(config, DecodeConfig):
        raise ValueError("需要字符串 prompt 与 DecodeConfig")
    if type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError("max_new_tokens 必须是非负整数")
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    mapping = {token: i for i, token in enumerate(vocab)}
    unknown = sorted(set(prompt) - set(mapping))
    if unknown:
        raise ValueError(f"提示含词表外字符 {unknown}；本课不静默替换提示")
    ids = [BOS_ID] + encode(prompt, mapping)
    if len(ids) > model.cfg.max_len:
        raise ValueError("提示加 BOS 已超出上下文，请缩短提示")
    device = next(model.parameters()).device
    rng = torch.Generator(device=device).manual_seed(seed)
    # 精确保留每个子模块的 train/eval 状态，而不只保留根模块状态。
    modes = [(module, module.training) for module in model.modules()]
    generated, trace, reason = [], [], "max_new_tokens"
    try:
        model.eval()
        with torch.no_grad():
            for step in range(max_new_tokens):
                if len(ids) > model.cfg.max_len:
                    reason = "context_limit"
                    break
                inputs = torch.tensor([ids], dtype=torch.long, device=device)
                next_logits = model(inputs)[0, -1, :]  # 必须取最后位置
                next_id, info = choose_next(next_logits, config, generator=rng)
                generated.append(next_id)
                ids.append(next_id)
                trace.append({"step": step, "token_id": next_id, "token": vocab[next_id], **info})
                if next_id == EOS_ID:
                    reason = "eos"
                    break
    finally:
        for module, training in modes:
            module.training = training
    visible_ids = [i for i in generated if i != EOS_ID]
    completion = "".join(vocab[i] for i in visible_ids)
    return {"prompt": prompt, "completion": completion, "text": prompt + completion,
            "generated_ids": generated, "visible_ids": visible_ids,
            "stop_reason": reason, "seed": seed, "max_new_tokens": max_new_tokens,
            "config": asdict(config), "trace": trace}

## 9. 评测：先定义指标，再看样例

默认设置 **3 个提示 × 每个提示 8 个 seed × 7 种策略 = 168 次生成**。
所有策略共用权重、提示列表、seed 列表和新增长度上限；不为某种策略单独挑选更好的模型。

| 指标 | 本课的精确定义 | 不能说明什么 |
|---|---|---|
| EOS 结束比例 | stop_reason 为 eos 的次数 / 生成次数 | 不代表语法、事实或推理正确 |
| 不同续写比例 | 同一提示下不同 completion 数 / 该提示抽样数，再对提示平均 | 不等于任务成功率；受样本数影响 |
| 模板成功比例 | 完整输出属于已知96条模板，且由 EOS 结束 | 不是通用语言质量或独立测试成绩 |
| 平均新增长度 | 新增编号总数（含 EOS）/ 次数 | 更长不代表更好 |
| distinct-2 | 所有 completion 中不同 bigram 数 / 总 bigram 数 | 受长度、样本量影响，不能单独优化 |
| 单样本重复率 | 每条中 $1-\frac{不同bigram数}{bigram总数}$，再平均 | 正常重复也可能被计入 |

多样性只统计 **completion**，不把固定提示计入；不跨样本拼接 bigram；没有 bigram 时 distinct-2 为 `None`。
贪心每提示生成8次但只有1种，比例是 **1/8=12.5%**，不是0%。每提示的不同数量也会保存在报告。
模板集合含训练文本，不是未见数据考核。EOS 与模板分数分开记录，避免把提前结束当成功。

自定义提示会关闭模板指标，显示为不适用。没有凭空为任意文字构造“正确率”。

In [20]:
def default_strategies() -> dict[str, DecodeConfig]:
    return {
        "greedy": DecodeConfig(mode="greedy"),
        "temperature_0.5": DecodeConfig(temperature=0.5),
        "temperature_1.0": DecodeConfig(),
        "temperature_1.8": DecodeConfig(temperature=1.8),
        "top_k_3": DecodeConfig(top_k=3),
        "top_p_0.9": DecodeConfig(top_p=0.9),
        "combined": DecodeConfig(temperature=0.8, top_k=5, top_p=0.9),
    }


def bigrams(ids: list[int]) -> list[tuple[int, int]]:
    return list(zip(ids[:-1], ids[1:]))


def summarize_generations(rows: list[dict], valid_templates: set[str] | None = None) -> dict:
    """每次只汇总同一策略的数据。只统计 completion，不把相同提示算进多样性。"""
    if not rows:
        raise ValueError("没有可汇总的生成记录")
    prompts = list(dict.fromkeys(row["prompt"] for row in rows))
    per_prompt = []
    all_bigrams, repeat_rates = [], []
    for prompt in prompts:
        group = [row for row in rows if row["prompt"] == prompt]
        unique_count = len({tuple(row["visible_ids"]) for row in group})
        per_prompt.append({"prompt": prompt, "samples": len(group),
                           "unique_count": unique_count, "unique_rate": unique_count / len(group)})
    for row in rows:
        grams = bigrams(row["visible_ids"])
        all_bigrams.extend(grams)  # 不连接不同样本，不跨样本制造 bigram。
        repeat_rates.append(1 - len(set(grams)) / len(grams) if grams else 0.0)
    result = {
        "samples": len(rows),
        "eos_rate": sum(row["stop_reason"] == "eos" for row in rows) / len(rows),
        "mean_new_tokens_including_eos": sum(len(row["generated_ids"]) for row in rows) / len(rows),
        "unique_rate_macro_by_prompt": sum(g["unique_rate"] for g in per_prompt) / len(per_prompt),
        "distinct_2_completion": len(set(all_bigrams)) / len(all_bigrams) if all_bigrams else None,
        "mean_within_sample_bigram_repeat_rate": sum(repeat_rates) / len(rows),
        "template_success_rate": (sum(row["text"] in valid_templates and row["stop_reason"] == "eos"
                                      for row in rows) / len(rows)) if valid_templates is not None else None,
        "per_prompt": per_prompt,
    }
    return result


def model_fingerprint(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str((tuple(tensor.shape), tensor.dtype)).encode("utf-8"))
        digest.update(bytes(tensor.reshape(-1).view(torch.uint8).tolist()))
    return digest.hexdigest()


def compare_strategies(model: TinyGPT, vocab: list[str], prompts: list[str], *,
                       samples_per_prompt: int = 8, seed_start: int = 1800,
                       max_new_tokens: int = 24, template_check: bool = True) -> tuple[list, dict]:
    positive_int(samples_per_prompt, "samples_per_prompt")
    if not prompts or any(not isinstance(p, str) for p in prompts) or len(set(prompts)) != len(prompts):
        raise ValueError("prompts 需要是非空、互不重复的字符串列表")
    if type(seed_start) is not int or not 0 <= seed_start <= 2**32 - samples_per_prompt:
        raise ValueError("seed 范围越界")
    valid_templates = set(demo_lines()) if template_check else None
    all_rows, report = [], {}
    before = model_fingerprint(model)
    for name, config in default_strategies().items():
        rows = []
        for prompt in prompts:
            for sample_index in range(samples_per_prompt):
                seed = seed_start + sample_index
                row = generate_text(model, vocab, prompt, config, max_new_tokens=max_new_tokens, seed=seed)
                row["strategy"] = name
                rows.append(row)
        all_rows.extend(rows)
        report[name] = summarize_generations(rows, valid_templates)
        summary = report[name]
        match = summary["template_success_rate"]
        match_text = "不适用" if match is None else f"{match:.1%}"
        print(f"{name:18s} | EOS={summary['eos_rate']:.1%} | 唯一比例={summary['unique_rate_macro_by_prompt']:.1%} | 模板成功={match_text}")
    after = model_fingerprint(model)
    if before != after:
        raise AssertionError("解码对照修改了模型权重！")
    return all_rows, {"by_strategy": report, "fingerprint_before": before, "fingerprint_after": after,
                      "weights_unchanged": True}

## 10. 训练一次，或载入已有检查点

默认训练一次，然后选取验证损失最低的一份模型，所有策略都用这一份。
设置 `CHECKPOINT=Path("lesson_17_outputs/best_model.pt")` 可跳过训练；路径相对于上面打印的当前工作目录。
只加载自己生成或信任的文件。`weights_only=True` 不意味着任意陌生文件都安全。

输出文件写入 `OUTPUT_DIR`；同名生成文件会覆盖。输入检查点不能和输出 `frozen_model.pt` 同路径。
脚本与 Notebook 使用同一实现。训练配方与第17课保持一致，但不同环境下不要求复现完全相同的选轮结果。

In [21]:
def train_demo_model(device: torch.device, *, epochs: int = 40, seed: int = 17) -> tuple[TinyGPT, list, dict]:
    positive_int(epochs, "epochs")
    train_texts, val_texts = split_texts(demo_lines(), seed)
    mapping, vocab = build_vocab(train_texts)
    cfg = GPTConfig(len(vocab))
    train_data = CharDataset(train_texts, mapping, cfg.max_len)
    val_data = CharDataset(val_texts, mapping, cfg.max_len)
    train_loader = make_loader(train_data, 16, True, seed)
    train_eval = make_loader(train_data, 16)
    val_loader = make_loader(val_data, 16)
    set_seed(seed)
    model = TinyGPT(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    # 与第17课保持一致的初始化、数据遍历及验证选择。
    before = {"train": evaluate(model, train_eval, device), "validation": evaluate(model, val_loader, device)}
    best_loss, best_epoch, best_state = before["validation"]["loss"], 0, snapshot_state(model)
    history = []
    for epoch in range(1, epochs + 1):
        online = train_epoch(model, train_loader, optimizer, device)
        train_metrics = evaluate(model, train_eval, device)
        validation = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "train": train_metrics, "validation": validation,
                        "online_train": online})
        if validation["loss"] < best_loss:
            best_loss, best_epoch, best_state = validation["loss"], epoch, snapshot_state(model)
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"准备固定模型 epoch={epoch:02d} train={train_metrics['loss']:.4f} val={validation['loss']:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    # 清掉训练残留梯度；比较过程不再创建/更新它们。
    model.zero_grad(set_to_none=True)
    return model, vocab, {"source": "embedded_demo_training", "epochs": epochs, "seed": seed,
                          "train_texts": train_texts, "validation_texts": val_texts,
                          "before": before, "best_epoch": best_epoch, "history": history}


def run_experiment(output_dir: Path, *, checkpoint: Path | None = None,
                   device: torch.device = torch.device("cpu"), epochs: int = 40,
                   samples_per_prompt: int = 8, seed_start: int = 1800,
                   max_new_tokens: int = 24, prompts: list[str] | None = None) -> dict:
    output_dir = Path(output_dir)
    output_names = ["frozen_model.pt", "training.json", "metrics.json", "generations.json", "experiment.json"]
    if checkpoint is not None and any(Path(checkpoint).resolve() == (output_dir / n).resolve() for n in output_names):
        raise ValueError("输入检查点不能与本次输出同路径，请换 --output-dir")
    if checkpoint is None:
        model, vocab, training_info = train_demo_model(device, epochs=epochs)
        source = "train_once_then_freeze"
    else:
        model, vocab, training_info = load_checkpoint(Path(checkpoint), device)
        training_info.update({"source": str(Path(checkpoint).resolve()), "training_skipped": True})
        source = "loaded_checkpoint"
    default_prompts = prompts is None
    if prompts is None:
        prompts = ["我今天正在", "小明明天正在", "老师今天正在"]
    # 先验证所有提示，长度预算为0，不消耗采样随机数。
    for prompt in prompts:
        generate_text(model, vocab, prompt, DecodeConfig(mode="greedy"), max_new_tokens=0)
    # 固定文本上的 teacher-forced 原始模型损失：只是“不改权重”的参照，不叫测试成绩。
    mapping = {token: i for i, token in enumerate(vocab)}
    probe_texts = [t for t in demo_lines() if len(t) + 1 <= model.cfg.max_len and all(c in mapping for c in t)]
    probe_source = "supported_demo_templates_including_training_overlap"
    if not probe_texts:
        probe_texts = prompts
        probe_source = "provided_prompt_strings_with_eos_not_heldout"
    probe_loader = make_loader(CharDataset(probe_texts, mapping, model.cfg.max_len), 16)
    nll_before = evaluate(model, probe_loader, device)
    samples, comparison = compare_strategies(model, vocab, prompts, samples_per_prompt=samples_per_prompt,
                                            seed_start=seed_start, max_new_tokens=max_new_tokens,
                                            template_check=default_prompts)
    nll_after = evaluate(model, probe_loader, device)
    if not math.isclose(nll_before["loss"], nll_after["loss"], rel_tol=1e-7, abs_tol=1e-7):
        raise AssertionError("固定权重和参照文本上的原始损失发生变化")
    output_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(output_dir / "frozen_model.pt", model, vocab, training_info.get("best_epoch", 0))
    reloaded, restored_vocab, _ = load_checkpoint(output_dir / "frozen_model.pt", device)
    probe_ids = torch.tensor([[BOS_ID] + encode(prompts[0], mapping)], device=device)
    with torch.no_grad():
        reload_difference = float((model(probe_ids) - reloaded(probe_ids)).abs().max().item())
    if restored_vocab != vocab or model_fingerprint(model) != model_fingerprint(reloaded):
        raise AssertionError("模型重载后参数或词表不一致")
    report = {"lesson": 18, "torch_version": str(torch.__version__), "device": str(device),
              "model_source": source, "model_config": asdict(model.cfg), "parameters": count_parameters(model),
              "prompts": prompts, "seeds": list(range(seed_start, seed_start + samples_per_prompt)),
              "samples_per_prompt": samples_per_prompt, "max_new_tokens": max_new_tokens,
              "strategies": {name: asdict(cfg) for name, cfg in default_strategies().items()},
              "template_metric_enabled": default_prompts, "generation_count": len(samples),
              "comparison": comparison,
              "fixed_text_probe": {"source": probe_source, "texts": probe_texts, "before": nll_before, "after": nll_after},
              "reload_max_abs_diff": reload_difference,
              "limitations": ["模板机制实验，不是独立测试成绩或真实任务正确率",
                              "同一固定前缀的熵比较与自由生成的汇总指标不是同一件事",
                              "同样seed跨设备、库版本不保证逐位一致；不同seed不保证不同输出",
                              "无KV cache；小词表float64过滤；不做性能/吞吐基准",
                              "仅预设7个配置；没有搜索最优参数；生成样本量不足以支持普适排名"]}
    for filename, obj in [("training.json", training_info), ("metrics.json", comparison["by_strategy"]),
                          ("generations.json", samples), ("experiment.json", report)]:
        (output_dir / filename).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    print(f"固定文本原始loss：{nll_before['loss']:.6f} → {nll_after['loss']:.6f}")
    print(f"权重未改变：{comparison['weights_unchanged']} | 保存记录：{len(samples)} 条")
    print("输出目录：", output_dir.resolve())
    return report

In [22]:
CHECKPOINT = None  # 已有第17课结果时改为 Path("lesson_17_outputs/best_model.pt")
OUTPUT_DIR = Path("lesson_18_outputs")
PROMPTS = None  # 默认3个教学提示；自定义例子：["我今天正在"]
report = run_experiment(
    OUTPUT_DIR, checkpoint=CHECKPOINT, device=DEVICE,
    epochs=40, samples_per_prompt=8, seed_start=1800,
    max_new_tokens=24, prompts=PROMPTS,
)

准备固定模型 epoch=01 train=2.5748 val=2.6290


准备固定模型 epoch=10 train=0.4321 val=0.4717


准备固定模型 epoch=20 train=0.3962 val=0.4369


准备固定模型 epoch=30 train=0.3911 val=0.4255


准备固定模型 epoch=40 train=0.3896 val=0.4504


greedy             | EOS=100.0% | 唯一比例=12.5% | 模板成功=100.0%


... (2 outputs left out)

temperature_1.8    | EOS=100.0% | 唯一比例=100.0% | 模板成功=12.5%


top_k_3            | EOS=100.0% | 唯一比例=37.5% | 模板成功=100.0%


top_p_0.9          | EOS=100.0% | 唯一比例=62.5% | 模板成功=100.0%


combined           | EOS=100.0% | 唯一比例=62.5% | 模板成功=100.0%
固定文本原始loss：0.399605 → 0.399605
权重未改变：True | 保存记录：168 条
输出目录： /mnt/data/_lesson18_build/notebook_run/lesson_18_outputs


## 11. 阅读结果：不要只挑好看的句子

下面显示**每种策略的第一个提示、前3个固定seed**，没有按是否成功筛选。
完整记录保存在 `generations.json`，可查看所有失败文本和停止原因。

一次有限实验不能给出所有任务的最优温度；它只能说明这组固定条件下的行为。
默认模板每个固定前缀只有6种合法活动，8次合法采样不可能得到8种不同合法答案，
所以更高的“不同输出比例”很可能已经包含错误文本。

In [23]:
all_samples = json.loads((OUTPUT_DIR / "generations.json").read_text(encoding="utf-8"))
for name in report["strategies"]:
    print()
    print(name)
    subset = [r for r in all_samples if r["strategy"] == name and r["prompt"] == report["prompts"][0]]
    for row in subset[:3]:
        print(row["seed"], repr(row["text"]), "停止：", row["stop_reason"])
print("报告记录数：", len(all_samples))


greedy
1800 '我今天正在学习数学。' 停止： eos
1801 '我今天正在学习数学。' 停止： eos
1802 '我今天正在学习数学。' 停止： eos

temperature_0.5
1800 '我今天正在整理笔记。' 停止： eos
1801 '我今天正在整理笔记。' 停止： eos
1802 '我今天正在编写程序。' 停止： eos

temperature_1.0
1800 '我今天正在整理笔记。' 停止： eos
1801 '我今天正在整理笔记。' 停止： eos
1802 '我今天正在编写程序。' 停止： eos

temperature_1.8
1800 '我今天正在整理笔记练模' 停止： eos
1801 '我今天正在整理笔查数。' 停止： eos
1802 '我今天正在编写正查你今今天正序。' 停止： eos

top_k_3
1800 '我今天正在整理笔记。' 停止： eos
1801 '我今天正在整理笔记。' 停止： eos
1802 '我今天正在编写程序。' 停止： eos

top_p_0.9
1800 '我今天正在整理笔记。' 停止： eos
1801 '我今天正在整理笔记。' 停止： eos
1802 '我今天正在编写程序。' 停止： eos

combined
1800 '我今天正在整理笔记。' 停止： eos
1801 '我今天正在整理笔记。' 停止： eos
1802 '我今天正在编写程序。' 停止： eos
报告记录数： 168


### 同一固定前缀的分布，与整段自由生成的统计，要分开观察

下面载入被冻结的模型，只计算一次相同前缀的原始 logits，再比较各策略的候选数、熵和前几个候选。
这才是受控的“同一个分布如何被变换”的对比。
日志里的 `chosen_probability` 是**解码变换后的选择概率**，不是事实置信度，也不一定是原始模型概率。

In [24]:
frozen_model, frozen_vocab, _ = load_checkpoint(OUTPUT_DIR / "frozen_model.pt", DEVICE)
prompt = report["prompts"][0]
mapping = {token: i for i, token in enumerate(frozen_vocab)}
with torch.no_grad():
    raw_logits = frozen_model(torch.tensor([[BOS_ID] + encode(prompt, mapping)], device=DEVICE))[0, -1]
for name, cfg in default_strategies().items():
    allowed = ban_tokens(raw_logits, (BOS_ID, PAD_ID))
    if cfg.mode == "greedy":
        probs = torch.zeros_like(allowed)
        probs[allowed.argmax()] = 1.
    else:
        probs = sampling_probs(allowed, temperature=cfg.temperature, top_k=cfg.top_k, top_p=cfg.top_p)
    ranked = torch.argsort(probs, descending=True, stable=True)
    top = [(frozen_vocab[i], round(float(probs[i]), 4)) for i in ranked[:6].tolist() if probs[i] > 0]
    print(name, "候选数=", int((probs>0).sum()), "熵=", round(distribution_entropy(probs),4), top)

greedy 候选数= 1 熵= -0.0 [('学', 1.0)]
temperature_0.5 候选数= 39 熵= 1.6694 [('学', 0.3523), ('整', 0.1793), ('编', 0.1534), ('检', 0.1289), ('训', 0.0966), ('阅', 0.0895)]
temperature_1.0 候选数= 39 熵= 1.8223 [('学', 0.2476), ('整', 0.1767), ('编', 0.1634), ('检', 0.1498), ('训', 0.1296), ('阅', 0.1248)]
temperature_1.8 候选数= 39 熵= 2.3535 [('学', 0.1844), ('整', 0.1528), ('编', 0.1464), ('检', 0.1394), ('训', 0.1287), ('阅', 0.126)]
top_k_3 候选数= 3 熵= 1.0814 [('学', 0.4213), ('整', 0.3006), ('编', 0.278)]
top_p_0.9 候选数= 6 熵= 1.7631 [('学', 0.2496), ('整', 0.1781), ('编', 0.1647), ('检', 0.151), ('训', 0.1307), ('阅', 0.1258)]
combined 候选数= 5 熵= 1.569 [('学', 0.3097), ('整', 0.2031), ('编', 0.1842), ('检', 0.1652), ('训', 0.1379)]


## 12. 原始模型损失为什么没有变化？

给定同一个固定文本，使用未调温、未截断的原始模型概率计算：
$$\mathrm{NLL}=-\frac1N\sum_t\log p_\theta(x_t\mid x_{<t}).$$
PPL 是平均 NLL 的指数。[6] 模型和参照文本不变，这个指标不需要经过任何生成策略，自然也不因解码设置改变。
如果把调温、截断后的概率用于打分，或者换成每组自己生成的文本，计算的对象已经改变，不能再叫“同一模型在同一数据上的原始损失”。

本课参照文本是能被词表表示的教学模板，**包含训练重合，不是独立测试集**。
外部模型词表不支持这些模板时，退回使用提供的提示文本作为数值参照；文件中记录具体来源。

In [25]:
probe = report["fixed_text_probe"]
print("参照来源：", probe["source"])
print("原始模型NLL，前/后：", probe["before"]["loss"], probe["after"]["loss"])
print("权重指纹一致：", report["comparison"]["weights_unchanged"])
print("重载最大输出差异：", report["reload_max_abs_diff"])
print("参数量：", report["parameters"])
print("同一权重哈希：", report["comparison"]["fingerprint_before"][:24])

参照来源： supported_demo_templates_including_training_overlap
原始模型NLL，前/后： 0.39960502056365316 0.39960502056365316
权重指纹一致： True
重载最大输出差异： 0.0
参数量： 109440
同一权重哈希： 2556bf57e99e70ac0935efff


## 13. 固定检查

下面的检查覆盖手算概率、温度边界、Top-k 并列、Top-p 阈值、过滤顺序、批量行独立、屏蔽候选、
局部随机源、EOS、零预算、上下文边界、错误路径的模式恢复、已有梯度保留、模型重载和评测分母。
这些检查不是性能基准，也不是语言能力测评。

In [26]:
class Checks:
    def __init__(self) -> None:
        self.count = 0

    def ok(self, condition: bool, message: str) -> None:
        if not bool(condition):
            raise AssertionError(message)
        self.count += 1

    def close(self, actual: torch.Tensor, expected: torch.Tensor, message: str,
              atol: float = 1e-9, rtol: float = 1e-7) -> None:
        torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol, msg=message)
        self.count += 1

    def raises(self, function, message: str, kind=ValueError) -> None:
        try:
            function()
        except kind:
            self.count += 1
        else:
            raise AssertionError(message)


class FixedLogitModel(nn.Module):
    """仅用于单元测试，不学习语言；每个位置返回预定分数。"""
    def __init__(self, vocab_size: int, max_len: int = 8, next_id: int = EOS_ID,
                 position_sensitive: bool = False, fail: bool = False) -> None:
        super().__init__()
        self.cfg = GPTConfig(vocab_size, max_len=max_len, d_model=8, n_heads=2, n_layers=1, ff_dim=16)
        self.anchor = nn.Parameter(torch.zeros(()))
        self.next_id = next_id
        self.position_sensitive = position_sensitive
        self.fail = fail
        self.calls = 0

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        if self.fail:
            raise RuntimeError("测试故意触发的模型错误")
        z = torch.full((*ids.shape, self.cfg.vocab_size), -1000., device=ids.device)
        z[..., self.next_id] = 0.
        if self.position_sensitive and ids.shape[1] > 1:
            z[:, 0, :] = -1000.
            z[:, 0, 4] = 0.
        return z + self.anchor * 0.


def run_checks(device: torch.device = torch.device("cpu")) -> int:
    c = Checks()
    def vec(data):
        return torch.tensor(data, dtype=torch.float64, device=device)

    probabilities = vec([0.5, 0.3, 0.15, 0.05])
    z = probabilities.log()
    c.close(sampling_probs(z), probabilities, "温度1且不截断应还原原始分布")
    c.close(sampling_probs(z + 1234), probabilities, "整体平移不应改变softmax")
    c.close(sampling_probs(z, temperature=0.5), probabilities.square() / probabilities.square().sum(), "温度必须除在logits上")
    c.close(sampling_probs(z, temperature=2), probabilities.sqrt() / probabilities.sqrt().sum(), "高温概率错误")
    c.ok(distribution_entropy(sampling_probs(z, temperature=.5)) < distribution_entropy(probabilities)
         < distribution_entropy(sampling_probs(z, temperature=2)), "固定前缀熵顺序错误")
    c.ok(int(sampling_probs(z, temperature=.01).argmax()) == int(z.argmax()), "正温度不应改变最大值索引")
    c.ok(bool(torch.isfinite(sampling_probs(vec([1e300, 0, -1e300]), temperature=1e-10)).all()), "极值稳定性失败")
    c.ok(torch.equal(z, probabilities.log()), "过滤不能原地修改输入")
    batch = torch.stack([z, z.flip(0)])
    c.close(sampling_probs(batch).sum(-1), vec([1, 1]), "应沿词表轴归一化")
    c.close(sampling_probs(batch)[1], probabilities.flip(0), "批量行串扰")

    c.close(sampling_probs(z, top_k=2), vec([.625, .375, 0, 0]), "Top-k应重归一化")
    c.close(top_k_filter(z, 0), z, "k=0应禁用")
    c.close(top_k_filter(z, 100), z, "k大于词表应保留全部")
    c.ok(int(torch.isfinite(top_k_filter(vec([0,0,0]), 2)).sum()) == 2, "并列时必须精确保留k个")
    c.ok(torch.equal(torch.isfinite(top_k_filter(vec([0,0,0]), 2)), torch.tensor([True,True,False],device=device)), "并列策略应按较小编号优先")
    c.ok(int(torch.isfinite(top_k_filter(vec([0,-math.inf,-math.inf]), 2)).sum()) == 1, "不能复活已屏蔽候选")
    c.close(sampling_probs(z, top_k=1), vec([1,0,0,0]), "唯一最大值时k=1退化为贪心")
    c.ok(torch.equal(torch.isfinite(top_k_filter(batch,2)).sum(-1), torch.tensor([2,2],device=device)), "Top-k批量错误")

    c.close(sampling_probs(z, top_p=.7), vec([.625,.375,0,0]), "必须保留跨阈值候选")
    c.close(sampling_probs(vec([0,0]),top_p=.5), vec([1,0]), "恰好达到阈值时不应再多保留一个")
    c.close(sampling_probs(z,top_p=1),probabilities,"p=1应禁用")
    c.close(sampling_probs(z,top_p=1e-12),vec([1,0,0,0]),"至少保留一个候选")
    c.close(sampling_probs(vec([0,-math.inf]),top_p=.8),vec([1,0]),"Top-p不能恢复被屏蔽项")
    c.close(sampling_probs(batch,top_p=.7),torch.stack([vec([.625,.375,0,0]),vec([0,0,.375,.625])]),"Top-p排序后必须还原token索引")
    c.close(sampling_probs(z,top_k=2,top_p=.6),vec([1,0,0,0]),"组合顺序必须先k再p：k后第一项概率.625")
    c.ok(int((sampling_probs(z,temperature=.4,top_p=.7)>0).sum()) == 1, "Top-p应使用调温后的分布")
    c.ok(int((sampling_probs(z,temperature=2,top_p=.7)>0).sum()) == 3, "温度升高可扩大核集合")
    c.close(sampling_probs(z,forbidden_ids=(0,)),vec([0,.6,.3,.1]),"禁止token后要重新归一化")

    for value in [0, -1, float('nan'), float('inf'), True, "1"]:
        c.raises(lambda value=value: DecodeConfig(temperature=value), "错误温度应拒绝")
    for value in [-1, True, 1.5]:
        c.raises(lambda value=value: DecodeConfig(top_k=value), "错误k应拒绝")
    for value in [0, -1, 1.1, float('nan'), True]:
        c.raises(lambda value=value: DecodeConfig(top_p=value), "错误p应拒绝")
    c.raises(lambda: DecodeConfig(mode="beam"),"未实现模式应拒绝")
    c.raises(lambda: DecodeConfig(mode="greedy", temperature=.5),"禁止静默忽略贪心采样参数")
    for invalid in [vec([]),vec([math.nan,0]),vec([math.inf,0]),vec([-math.inf,-math.inf]),
                    torch.tensor([1,2],device=device),torch.zeros(1,1,2,device=device)]:
        c.raises(lambda invalid=invalid:sampling_probs(invalid),"非法logits未拒绝")
    c.raises(lambda: sampling_probs(z, forbidden_ids=(0,1,2,3)), "不能屏蔽全部候选")
    c.raises(lambda: sampling_probs(z, forbidden_ids=(4,)), "禁止编号越界")
    c.raises(lambda: sampling_probs(z, forbidden_ids=(True,)), "bool不是合法token编号")

    g1 = torch.Generator(device=device).manual_seed(1)
    g2 = torch.Generator(device=device).manual_seed(1)
    draws1 = torch.multinomial(probabilities, 32, replacement=True, generator=g1)
    draws2 = torch.multinomial(probabilities, 32, replacement=True, generator=g2)
    c.ok(torch.equal(draws1, draws2), "同环境同局部随机源不能复现")
    g3 = torch.Generator(device=device).manual_seed(4)
    state = g3.get_state().clone()
    next_id, info = choose_next(z, DecodeConfig(mode="greedy"), g3, forbidden_ids=())
    c.ok(next_id == 0 and torch.equal(state,g3.get_state()), "贪心不应消耗采样随机数")
    c.ok(info['entropy_nats'] == 0 and info['support_size'] == 1, "贪心选择分布应为one-hot")
    filtered_draws = torch.multinomial(sampling_probs(z,top_k=2),128,replacement=True,generator=g3)
    c.ok(bool((filtered_draws < 2).all()),"被屏蔽候选不应被采样")

    _, vocab = build_vocab(["我爱学习。", "你爱模型。"])
    greedy = DecodeConfig(mode="greedy")
    fixed = FixedLogitModel(len(vocab)).to(device)
    row = generate_text(fixed,vocab,"我",greedy,max_new_tokens=8)
    c.ok(row['generated_ids'] == [EOS_ID] and row['stop_reason']=='eos', "EOS应立即停止且计入编号")
    c.ok(row['completion']=='' and row['text']=='我' and fixed.calls==1, "不能在EOS后继续前向")
    fixed.calls=0
    zero = generate_text(fixed,vocab,"",greedy,max_new_tokens=0)
    c.ok(zero['generated_ids']==[] and zero['stop_reason']=='max_new_tokens' and fixed.calls==0,"零预算应零次前向")
    fixed = FixedLogitModel(len(vocab),max_len=4,next_id=5).to(device)
    c.ok(len(generate_text(fixed,vocab,"",greedy,max_new_tokens=2)['generated_ids'])==2,"预算计算不能包含prompt")
    bound = generate_text(fixed,vocab,"我我我",greedy,max_new_tokens=3)
    c.ok(bound['stop_reason']=='context_limit' and len(bound['generated_ids'])==1,"上下文边界应停止，不截断旧token")
    c.raises(lambda:generate_text(fixed,vocab,"我我我我",greedy),"超长提示必须拒绝")
    c.raises(lambda:generate_text(fixed,vocab,"火",greedy),"词表外提示必须拒绝")
    c.raises(lambda:generate_text(fixed,vocab,"",greedy,seed=-1),"负seed应拒绝")
    c.raises(lambda:generate_text(fixed,vocab,"",greedy,max_new_tokens=-1),"负预算应拒绝")
    positional=FixedLogitModel(len(vocab),next_id=5,position_sensitive=True).to(device)
    c.ok(generate_text(positional,vocab,"我",greedy,max_new_tokens=1)['generated_ids']==[5],"必须取最后位置logits")
    forbidden_logits=torch.full((len(vocab),),-100.,device=device)
    forbidden_logits[BOS_ID]=100
    forbidden_logits[PAD_ID]=90
    forbidden_logits[EOS_ID]=10
    c.ok(choose_next(forbidden_logits,greedy)[0]==EOS_ID,"BOS/PAD必须在选择前屏蔽")
    failing=FixedLogitModel(len(vocab),fail=True).to(device).train()
    c.raises(lambda:generate_text(failing,vocab,"我",greedy),"错误应传播",RuntimeError)
    c.ok(failing.training,"异常路径也要恢复模式")

    set_seed(18)
    cfg=GPTConfig(len(vocab),max_len=16,d_model=16,n_heads=4,n_layers=2,ff_dim=32,dropout=.2)
    model=TinyGPT(cfg).to(device).train()
    model.blocks[0].eval()  # 混合模式，生成后必须逐个恢复。
    modes=[m.training for m in model.modules()]
    for parameter in model.parameters():
        parameter.grad=torch.ones_like(parameter)
    before=model_fingerprint(model)
    cpu_rng=torch.get_rng_state().clone()
    r1=generate_text(model,vocab,"我",DecodeConfig(top_p=.9),max_new_tokens=6,seed=123)
    c.ok([m.training for m in model.modules()]==modes,"子模块模式恢复失败")
    c.ok(model_fingerprint(model)==before,"生成修改了权重")
    c.ok(all(torch.equal(p.grad,torch.ones_like(p)) for p in model.parameters()),"生成修改了已有梯度")
    c.ok(torch.equal(cpu_rng,torch.get_rng_state()),"局部采样不应消耗全局CPU随机数")
    r2=generate_text(model,vocab,"我",DecodeConfig(top_p=.9),max_new_tokens=6,seed=123)
    c.ok(r1['generated_ids']==r2['generated_ids'],"生成同seed无法复现")
    g1=generate_text(model,vocab,"我",greedy,max_new_tokens=4,seed=1)
    g2=generate_text(model,vocab,"我",greedy,max_new_tokens=4,seed=2)
    c.ok(g1['generated_ids']==g2['generated_ids'],"贪心不应依赖seed")
    model.eval()
    ds=CharDataset(["我爱", "你"],{v:i for i,v in enumerate(vocab)},16)
    loader=make_loader(ds,2)
    loss0=evaluate(model,loader,device)['loss']
    generate_text(model,vocab,"我",DecodeConfig(temperature=1.8),max_new_tokens=4,seed=3)
    loss1=evaluate(model,loader,device)['loss']
    c.ok(loss0==loss1,"采样参数不能改变固定文本的原始模型loss")
    with tempfile.TemporaryDirectory() as temp:
        checkpoint=Path(temp)/'weights.pt'
        save_checkpoint(checkpoint,model,vocab,0)
        loaded,loaded_vocab,_=load_checkpoint(checkpoint,device)
        c.ok(model_fingerprint(loaded)==model_fingerprint(model) and loaded_vocab==vocab,"重载必须保留模型及词表")
        c.ok(generate_text(loaded,vocab,"我",greedy,max_new_tokens=3)['generated_ids'] ==
             generate_text(model,vocab,"我",greedy,max_new_tokens=3)['generated_ids'],"重载后的贪心输出应一致")

    def fake(prompt, ids, reason="eos", text="我今天正在学习数学。"):
        return {"prompt":prompt,"visible_ids":ids,"generated_ids":ids+[EOS_ID],"stop_reason":reason,"text":text}
    rows=[fake('甲',[4,5,4,5]),fake('甲',[4,5,4,5]),fake('乙',[6]),fake('乙',[7])]
    summary=summarize_generations(rows,set(demo_lines()))
    c.ok(summary['unique_rate_macro_by_prompt']==.75,"唯一比例应按提示分组")
    c.ok(math.isclose(summary['mean_within_sample_bigram_repeat_rate'],1/6),"重复bigram定义错误")
    c.ok(math.isclose(summary['distinct_2_completion'],2/6),"不能跨样本构造bigram或纳入prompt")
    c.ok(summary['template_success_rate']==1,"模板成功应检查完整文本")
    short=summarize_generations([fake('甲',[])])
    c.ok(short['distinct_2_completion'] is None,"没有bigram时不得编造多样性得分")
    c.ok(short['template_success_rate'] is None,"未提供规则集合时不计算模板成功")
    truncated=summarize_generations([fake('甲',[4],reason='max_new_tokens')],set(demo_lines()))
    c.ok(truncated['template_success_rate']==0,"文本匹配但未正常EOS不算模板完成")
    c.raises(lambda:summarize_generations([]),"空评测应拒绝")
    c.ok(.6*.51 < .4*.99,"贪心反例应说明局部最大不保证整段最大")
    print(f"通过固定检查：{c.count} 项")
    return c.count

In [27]:
check_count = run_checks(DEVICE)
print("检查数：", check_count)

通过固定检查：88 项
检查数： 88


## 14. 独立练习 A：自己实现 Top-k 概率

实现 `top_k_probs_ex(logits,k)`，返回重归一化后的概率，支持 [V] / [B,V]。
可以复用 `validate_logits()` 和 `DecodeConfig()` 做输入检查，但不能调用本课已有过滤或采样函数。
要求 k=0 禁用、k过大保留全部、并列时编号较小者优先，不能原地改输入。

先手算：分布 [0.5,0.3,0.15,0.05]，k=2，应得到什么？

In [28]:
def top_k_probs_ex(logits, k):
    raise NotImplementedError("请补全：检查、排序、保留、softmax")


def check_ex_a(function):
    p = torch.tensor([.5,.3,.15,.05], dtype=torch.float64)
    z = p.log()
    torch.testing.assert_close(function(z,2), torch.tensor([.625,.375,0,0],dtype=torch.float64))
    torch.testing.assert_close(function(z,0),p)
    torch.testing.assert_close(function(z,100),p)
    torch.testing.assert_close(function(torch.zeros(3),2),torch.tensor([.5,.5,0],dtype=torch.float64))
    batch=torch.stack([z,z.flip(0)])
    expected=torch.tensor([[.625,.375,0,0],[0,0,.375,.625]],dtype=torch.float64)
    torch.testing.assert_close(function(batch,2),expected)
    torch.testing.assert_close(z,p.log())
    print("练习A：6项检查通过")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_ex_a(top_k_probs_ex)

## 15. 独立练习 B：自己实现 Top-p 过滤

实现 `top_p_filter_ex(logits,p)`，返回过滤后的 logits，不是概率。
允许使用输入检查函数，不能调用 `top_p_filter()` 或 `sampling_probs()`。
处理排序、累计概率、跨阈值候选、散射回原编号、p=1 和批量维度。

提示：排序后的第 i 项是否保留，应看**它之前**的累计概率。

In [29]:
def top_p_filter_ex(logits, p):
    raise NotImplementedError("请补全：排序、累计、保留、还原编号")


def check_ex_b(function):
    z = torch.tensor([.15,.5,.05,.3],dtype=torch.float64).log()
    result = torch.softmax(function(z,.7),-1)
    torch.testing.assert_close(result,torch.tensor([0,.625,0,.375],dtype=torch.float64))
    torch.testing.assert_close(function(z,1),z)
    torch.testing.assert_close(torch.softmax(function(torch.zeros(2),.5),-1),torch.tensor([1.,0.],dtype=torch.float64))
    torch.testing.assert_close(torch.softmax(function(z,1e-9),-1),torch.tensor([0.,1.,0.,0.],dtype=torch.float64))
    batch=torch.stack([z,z.flip(0)])
    torch.testing.assert_close(torch.softmax(function(batch,.7),-1),torch.stack([result,result.flip(0)]))
    torch.testing.assert_close(z,torch.tensor([.15,.5,.05,.3],dtype=torch.float64).log())
    print("练习B：6项检查通过")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_ex_b(top_p_filter_ex)

## 16. 独立练习 C：定义一个不夸大效果的评测函数

实现 `summarize_ex(rows, valid_templates=None)`，不要调用现成的 `summarize_generations()`。
返回三个字段：`eos_rate`、`unique_rate_macro_by_prompt`、`template_success_rate`。
不同续写按 `visible_ids` 比较，分提示计算后平均；模板成功必须“完整文本匹配且 EOS 停止”；未提供模板集合时返回 None。

再做变化实验：把采样次数从8改成16，在**同一冻结检查点**上重跑，保存到新目录。
解释为什么不同续写比例可能下降，不能把它简单说成“模型退化”。

In [30]:
def summarize_ex(rows, valid_templates=None):
    raise NotImplementedError("请补全分组、EOS比例与模板成功规则")


def check_ex_c(function):
    rows = [
        {"prompt":"甲","visible_ids":[4],"text":"甲好","stop_reason":"eos"},
        {"prompt":"甲","visible_ids":[4],"text":"甲好","stop_reason":"max_new_tokens"},
        {"prompt":"乙","visible_ids":[5],"text":"乙好","stop_reason":"eos"},
        {"prompt":"乙","visible_ids":[6],"text":"乙坏","stop_reason":"eos"},
    ]
    result=function(rows,{"甲好","乙好"})
    assert result['eos_rate']==.75
    assert result['unique_rate_macro_by_prompt']==.75
    assert result['template_success_rate']==.5
    assert function(rows)['template_success_rate'] is None
    # 每个提示样本数不同时，也应按提示做macro平均，而非把不同提示揉成一个集合。
    extra=rows+[rows[0]]
    assert math.isclose(function(extra)['unique_rate_macro_by_prompt'],(1/3+1)/2)
    print("练习C：5项检查通过")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_ex_c(summarize_ex)

---
## 参考答案：先独立完成，再核对

下面不是把答案函数指向前面的现成实现，而是重新写出核心步骤。
如果你的实现不同，只要满足相同输入约定、边界行为和固定检查，也可能是正确的。

In [31]:
def top_k_probs_reference(logits, k):
    DecodeConfig(top_k=k)
    z = validate_logits(logits)
    if 0 < k < z.shape[-1]:
        order = torch.argsort(z, descending=True, stable=True, dim=-1)
        selected = order[..., :k]
        result = torch.full_like(z, -math.inf)
        result.scatter_(-1, selected, z.gather(-1, selected))
        z = result
    return torch.softmax(z, dim=-1)

check_ex_a(top_k_probs_reference)

练习A：6项检查通过


In [32]:
def top_p_filter_reference(logits, p):
    DecodeConfig(top_p=p)
    z = validate_logits(logits)
    if p == 1:
        return z
    values, indices = torch.sort(z, descending=True, stable=True, dim=-1)
    cumulative = torch.softmax(values, -1).cumsum(-1)
    preceding = torch.cat((torch.zeros_like(cumulative[..., :1]), cumulative[..., :-1]), -1)
    allowed = (preceding < p) & torch.isfinite(values)
    allowed[..., 0] = True
    result = torch.full_like(z, -math.inf)
    result.scatter_(-1, indices, values.masked_fill(~allowed, -math.inf))
    return result

check_ex_b(top_p_filter_reference)

练习B：6项检查通过


In [33]:
def summarize_reference(rows, valid_templates=None):
    if not rows:
        raise ValueError("没有可评估的记录")
    prompts = {row["prompt"] for row in rows}
    unique_ratios = []
    for prompt in prompts:
        group = [row for row in rows if row["prompt"] == prompt]
        completions = {tuple(row["visible_ids"]) for row in group}
        unique_ratios.append(len(completions) / len(group))
    eos_rate = sum(row["stop_reason"] == "eos" for row in rows) / len(rows)
    success = None
    if valid_templates is not None:
        success = sum(row["text"] in valid_templates and row["stop_reason"] == "eos" for row in rows) / len(rows)
    return {"eos_rate": eos_rate,
            "unique_rate_macro_by_prompt": sum(unique_ratios) / len(unique_ratios),
            "template_success_rate": success}

check_ex_c(summarize_reference)

练习C：5项检查通过


## 运行脚本与输出文件

```bash
python lesson_18_decoding_and_generation_evaluation.py
python lesson_18_decoding_and_generation_evaluation.py --checks-only
python lesson_18_decoding_and_generation_evaluation.py --checkpoint lesson_17_outputs/best_model.pt
```

已有CUDA可追加 `--device auto`；沿用已有环境，不要求显卡。
复用本课冻结模型做更多采样时，使用不同输出目录，避免覆盖作为输入的文件：

```bash
python lesson_18_decoding_and_generation_evaluation.py --checkpoint lesson_18_outputs/frozen_model.pt --samples-per-prompt 16 --output-dir lesson_18_more_samples
```

自定义提示可重复指定 `--prompt`，且字符必须在词表中。任意自定义提示会关闭内置模板得分。

| 文件 | 含义 |
|---|---|
| `frozen_model.pt` | 固定模型配置、词表和权重；与第17课推理格式兼容 |
| `training.json` | 本次准备训练记录，或加载检查点的来源信息 |
| `generations.json` | 全部生成结果、seed、停止原因和逐token采样诊断 |
| `metrics.json` | 逐策略汇总和逐提示的不同输出数量 |
| `experiment.json` | 配置、环境、权重指纹、固定文本loss核对与实验限制 |

导入脚本不会训练或写文件。运行实验会覆盖输出目录中的同名文件；不会修改输入检查点。

## 常见问题与本课边界

**温度更高但这次样例没变？** 随机抽样可能碰巧选中相同候选；先检查同一前缀的分布，再比较多次结果。

**Top-p=0.9 为什么只剩一个候选？** 某个位置的最高概率已经超过阈值；不是所有位置都需要多个候选。

**设置不同 seed 却都输出同一句？** 贪心本来不依赖seed；采样分布也可能非常尖锐。

**生成全部正常EOS，为什么模板成功率低？** 能停止不代表内容正确，不能把停止指标当质量指标。

**高温多样性上升，能力是否提高？** 要结合真实任务的正确性判断；本课只改变输出选择规则，权重没有更新。

**能直接照搬为生产推理模块吗？** 不建议：这里按单条前缀重算，小词表全排序、float64筛选和大量检查都偏教学。没有批量停止管理、KV cache、流式接口和吞吐优化。

本课没有束搜索、重复惩罚、工具调用、约束JSON解码；先把最常用的几个基础策略验证清楚。
下一课：**第19课，KV cache、Prefill 与 Decode**，保持相同模型计算含义，减少重复计算，并检验缓存前后logits是否一致。

## 资料与复现说明

[1] [Hugging Face：Generation strategies](https://huggingface.co/docs/transformers/generation_strategies)，贪心与采样。

[2] [Hugging Face：GenerationConfig](https://huggingface.co/docs/transformers/main_classes/text_generation)，温度、Top-k、Top-p 与长度参数。

[3] [Holtzman 等：The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751)，核采样的原始研究；论文实验结论不等于所有新模型/任务的最优解码建议。

[4] [PyTorch：Reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)，随机源与跨环境复现边界。

[5] [PyTorch：torch.multinomial](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)，按权重抽样。

[6] [Hugging Face：Perplexity](https://huggingface.co/docs/transformers/perplexity)，固定文本条件概率指标。

本文件保留实际运行输出。检查开关默认关闭的是独立练习，不是示例或固定检查。
环境信息以首个输出单元为准；当前交付的CPU运行结果不代表已在RTX 3060上实测。

**交付验证记录：** 当前环境 PyTorch 2.10.0+cpu；57 个单元中的 33 个代码单元已从新内核顺序执行，无错误。88 项固定检查通过。三个参考实现分别在独立命名空间运行通过。与独立脚本生成的冻结权重及全部168条生成记录一致；第17课保存的检查点可由本课模型逐位重现前向输出。六种故障注入（温度乘除颠倒、Top-p遗漏跨阈值项、丢失token索引、取错生成位置、遗漏禁止token、模板统计忽略EOS）均被现有检查发现。未在CUDA/RTX 3060上实测。